<a href="https://colab.research.google.com/github/AmrMaarouf/ITI/blob/main/titanic_kaggle_submission_Amr_Abdelfatah_Mahmoud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

titanic_kaggle_submission_Amr_Abdelfatah_Mahmoud.ipynb

In [2]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_4ed90de8c551fca1d9cd901ab4dadd56"

!pip install kaggle -q
!kaggle competitions download -c titanic
!unzip -o titanic.zip

titanic.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  titanic.zip
  inflating: gender_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.isnull().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [5]:
def preprocess(df, age_median, fare_median, embarked_mode):
    df = df.copy()
    df["Age"] = df["Age"].fillna(age_median)
    df["Fare"] = df["Fare"].fillna(fare_median)
    df["Embarked"] = df["Embarked"].fillna(embarked_mode)

    # Feature engineering
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)
    df["HasCabin"] = df["Cabin"].notnull().astype(int)

    # Encoding
    df["Sex"] = df["Sex"].map({"male": 1, "female": 0})
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    df = df.drop(columns=["Name", "Ticket", "Cabin"])
    return df

age_median = train["Age"].median()
fare_median = train["Fare"].median()
embarked_mode = train["Embarked"].mode()[0]

train_p = preprocess(train, age_median, fare_median, embarked_mode)
test_p = preprocess(test, age_median, fare_median, embarked_mode)

In [6]:
X = train_p.drop(columns=["Survived", "PassengerId"])
y = train_p["Survived"]
X_test_final = test_p.drop(columns=["PassengerId"])[X.columns]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train, y_train)
print("Baseline accuracy:", baseline.score(X_val, y_val))

Baseline accuracy: 0.8044692737430168


In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    cv_score = cross_val_score(model, X, y, cv=5).mean()
    print(f"{name}: CV accuracy = {cv_score*100:.2f}")

Logistic Regression: CV accuracy = 79.91
Random Forest: CV accuracy = 80.25
Gradient Boosting: CV accuracy = 82.61


In [10]:
final_model = GradientBoostingClassifier(random_state=42)
final_model.fit(X, y)

predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    "PassengerId": test_p["PassengerId"],
    "Survived": predictions
})

submission.to_csv("submission.csv", index=False)

In [11]:
!kaggle competitions submit -c titanic -f submission.csv -m "Gradient Boosting tuned model"

100% 2.77k/2.77k [00:00<00:00, 6.26kB/s]
Successfully submitted to Titanic - Machine Learning from Disaster

In [13]:
!kaggle competitions submissions -c titanic

fileName        date                        description                    status                     publicScore  privateScore  
--------------  --------------------------  -----------------------------  -------------------------  -----------  ------------  
submission.csv  2026-09-09 09:59:40.810000  Gradient Boosting tuned model  SubmissionStatus.COMPLETE  0.77751                    
